# Experiment: SpectralBio Final Accept Hardening Part 1

This notebook runs the lighter and more evidence-dense half of the final validation protocol.

## Scope
- Global ClinVar support scan
- Support-ranked multigene panel
- TP53 nested-CV leakage audit
- Shortlist nested CV on top non-anchor genes
- Recommendation of a second canonical benchmark candidate

## Intended runtime
- Good fit for Colab T4
- No 3B checkpoint here


In [ ]:
from pathlib import Path

USE_GOOGLE_DRIVE = False
DRIVE_MOUNT_POINT = Path('/content/drive')
DRIVE_OUTPUT_SUBDIR = Path('MyDrive/SpectralBioFinalAcceptSplit')

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount(str(DRIVE_MOUNT_POINT))
    print('Drive mounted at', DRIVE_MOUNT_POINT)
else:
    print('Drive mount skipped; outputs stay in the VM unless OUTPUT_ROOT is changed later.')


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/DaviBonetto/SpectralBio.git'
REPO_BRANCH = 'codex/claw4s-rebuild'
REPO_ROOT = Path('/content/Stanford-Claw4s')
BOOTSTRAP_MARKER = REPO_ROOT / '.colab_bootstrap_v3_complete'

if not REPO_ROOT.exists():
    subprocess.check_call(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(REPO_ROOT)])

os.chdir(REPO_ROOT)
subprocess.check_call(['git', 'fetch', 'origin', REPO_BRANCH])
subprocess.check_call(['git', 'checkout', REPO_BRANCH])
src_path = REPO_ROOT / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

if not BOOTSTRAP_MARKER.exists():
    subprocess.check_call([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchvision'])
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', '.', '--no-deps'])
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install',
        'numpy==2.1.3', 'plotly==5.24.1', 'pyyaml==6.0.2', 'scikit-learn==1.5.2',
        'scipy==1.14.1', 'transformers==4.49.0', 'pandas', 'matplotlib'
    ])
    BOOTSTRAP_MARKER.write_text('ok\n', encoding='utf-8')
    print('Dependencies installed. Restarting runtime once...')
    os.kill(os.getpid(), 9)
else:
    print('Bootstrap marker found; skipping reinstall.')


## Plan

- Use support-ranked rather than hand-picked gene expansion.
- Keep models light enough for T4.
- Produce the evidence needed to argue for a second benchmark candidate.


In [ ]:
import subprocess
import sys
import pandas as pd

try:
    from spectralbio.supplementary.final_accept_hardening import (
        AcceptHardeningConfig,
        build_support_ranked_panel_manifest,
        create_accept_hardening_paths,
        recommend_second_benchmark_candidate,
        run_shortlist_gene_nested_cv,
        run_support_ranked_panel,
        scan_clinvar_gene_support,
    )
    from spectralbio.supplementary.reject_recovery import (
        NestedCVConfig,
        create_reject_recovery_zip,
        run_tp53_nested_cv_audit,
        write_experiment_log,
    )
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', '.', '--no-deps'])
    from spectralbio.supplementary.final_accept_hardening import (
        AcceptHardeningConfig,
        build_support_ranked_panel_manifest,
        create_accept_hardening_paths,
        recommend_second_benchmark_candidate,
        run_shortlist_gene_nested_cv,
        run_support_ranked_panel,
        scan_clinvar_gene_support,
    )
    from spectralbio.supplementary.reject_recovery import (
        NestedCVConfig,
        create_reject_recovery_zip,
        run_tp53_nested_cv_audit,
        write_experiment_log,
    )

MIN_TOTAL = 60
MIN_PER_CLASS = 20
MAX_ADDITIONAL_GENES = 8
SHORTLIST_NON_ANCHOR_GENES = 3
BOOTSTRAP_REPLICATES = 1000
OVERWRITE = False

OUTPUT_ROOT = (
    DRIVE_MOUNT_POINT / DRIVE_OUTPUT_SUBDIR
    if USE_GOOGLE_DRIVE
    else REPO_ROOT / 'supplementary' / 'final_accept_split'
)

config = AcceptHardeningConfig(
    stronger_model_names=('facebook/esm2_t33_650M_UR50D',),
    min_total=MIN_TOTAL,
    min_per_class=MIN_PER_CLASS,
    max_additional_genes=MAX_ADDITIONAL_GENES,
    shortlist_non_anchor_genes=SHORTLIST_NON_ANCHOR_GENES,
    bootstrap_replicates=BOOTSTRAP_REPLICATES,
    overwrite=OVERWRITE,
)
paths = create_accept_hardening_paths(repo_root=REPO_ROOT, output_root=OUTPUT_ROOT)
nested_config = NestedCVConfig(
    n_splits=config.nested_cv_n_splits,
    n_repeats=config.nested_cv_n_repeats,
    alpha_step=config.alpha_step,
    random_seed=config.random_seed,
    render_figures=config.render_figures,
    overwrite=config.overwrite,
)
print(paths)
print(config)


In [ ]:
support_scan = scan_clinvar_gene_support(paths, config)
panel_manifest = build_support_ranked_panel_manifest(paths, config, support_scan)
tp53_nested = run_tp53_nested_cv_audit(paths, nested_config)
multigene_metrics = run_support_ranked_panel(paths, config, panel_manifest)
shortlist_nested = run_shortlist_gene_nested_cv(paths, config, panel_manifest, multigene_metrics)
candidate = recommend_second_benchmark_candidate(paths, config, panel_manifest, multigene_metrics, shortlist_nested)

print('Selected genes:', panel_manifest['selected_genes'])
print('Recommended second benchmark candidate:', candidate['recommended_gene'])


In [ ]:
support_table = pd.read_csv(paths.root / 'support_scan' / 'clinvar_gene_support_table.csv')
support_ranked = pd.read_csv(paths.multigene / 'support_ranked_gene_table.csv')
multigene_summary = pd.read_csv(paths.multigene / 'multigene_summary.csv')

display(support_table.head(20))
display(support_ranked)
display(multigene_summary.sort_values(['delta_pair_vs_ll', 'n_total'], ascending=[False, False]))
display(pd.DataFrame(candidate['candidates']))


In [ ]:
notes = [
    'Part 1 focuses on support-ranked panel evidence and benchmark recommendation.',
    f"Recommended second benchmark candidate: {candidate['recommended_gene']}",
]
log_path = write_experiment_log(
    paths,
    completed_experiments=[
        'Global ClinVar support scan',
        'Support-ranked panel build',
        'TP53 nested CV',
        'Support-ranked multigene panel',
        'Shortlist nested CV',
        'Second benchmark candidate recommendation',
    ],
    skipped_experiments=[],
    notes=notes,
)
zip_path = create_reject_recovery_zip(paths, bundle_name='spectralbio_final_accept_part1_bundle')
print('Experiment log:', log_path)
print('ZIP ready at:', zip_path)


In [ ]:
print(f'ZIP ready at: {zip_path}')

if zip_path.exists():
    from google.colab import files
    files.download(str(zip_path))
else:
    print('ZIP not found:', zip_path)
